In [46]:
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

In [41]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("adhamelkomy/bank-customer-complaint-analysis")

100%|██████████| 20.0M/20.0M [00:00<00:00, 182MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/adhamelkomy/bank-customer-complaint-analysis/versions/1


In [53]:
import os
import shutil

# Define the destination directory
destination_dir = 'data'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# Copy the contents of the downloaded dataset directory to the destination directory
# Assuming 'path' contains the directory where the dataset was extracted

# Get the base name of the downloaded folder (e.g., '1' from versions/1)
dataset_folder_name = os.path.basename(path)

# Construct the full path to the source directory within the cache
source_dir = path

for item in os.listdir(source_dir):
    s = os.path.join(source_dir, item)
    d = os.path.join(destination_dir, item)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    else:
        shutil.copy2(s, d)

print(f"Dataset copied from {path} to {destination_dir}/")
print(f"Contents of {destination_dir}: {os.listdir(destination_dir)}")

Dataset copied from /root/.cache/kagglehub/datasets/adhamelkomy/bank-customer-complaint-analysis/versions/1 to data/
Contents of data: ['complaints_report_20240226_183305.txt', 'Bank Customer Complaint Analysis for Efficient Dispute Resolution.ipynb', 'complaints.csv', 'final_dataframe (1).csv']


In [62]:
df = pd.read_csv("data/complaints.csv")
df.head(3)

,Unnamed: 0,product,narrative
0,0,credit_card,purchase order day shipping amount receive pro...
1,1,credit_card,forwarded message date tue subject please inve...
2,2,retail_banking,forwarded message cc sent friday pdt subject f...


In [65]:
# 2. Clean missing data
# Real narrative text often has blank spaces or missing values
df = df.drop_duplicates(subset=['narrative'], keep='first')

# 3. Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    df['narrative'],
    df['product'],
    test_size=0.2,
    random_state=42,
    stratify=df['product'] # Vital because some products have way more complaints
)

# 4. Build a Robust Natural Language Pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words='english',
        ngram_range=(1, 2),    # Captures phrases like "credit card" or "wells fargo"
        max_features=15000,    # Limits vocabulary size to filter out rare typos/names
        sublinear_tf=True      # Essential for long texts; dampens words repeated out of anger
    )),
    ('clf', LogisticRegression(
        multi_class='multinomial',
        solver='lbfgs',
        class_weight='balanced', # Automatically manages imbalanced class distributions
        max_iter=1000,
        random_state=42
    ))
])

# 5. Train the System
pipeline.fit(X_train, y_train)

# 6. Evaluate Real Metrics
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


                     precision    recall  f1-score   support

        credit_card       0.75      0.84      0.80      2997
   credit_reporting       0.93      0.83      0.88     11248
    debt_collection       0.76      0.82      0.79      4211
mortgages_and_loans       0.81      0.89      0.85      3745
     retail_banking       0.85      0.91      0.88      2694

           accuracy                           0.85     24895
          macro avg       0.82      0.86      0.84     24895
       weighted avg       0.85      0.85      0.85     24895



In [66]:
new_tickets = [
"I am trying to get a mortgage but my bank statement shows an unknown charge from last week.",
    "Experian refuses to remove a late payment history that wasn't mine."
]

predictions = pipeline.predict(new_tickets)
print("Predictions:", predictions)

Predictions: ['mortgages_and_loans' 'credit_reporting']


In [67]:
import joblib

# Save the trained pipeline file to disk
model_filename = 'complaints_classifier.joblib'
joblib.dump(pipeline, model_filename)
print(f"Model successfully saved to {model_filename}")

Model successfully saved to complaints_classifier.joblib


In [54]:
# Load the compiled model
pipeline = joblib.load('complaints_classifier.joblib')

# Pure, unseen user text
incoming_complaints = [
    "I am trying to get a mortgage but my bank statement shows an unknown charge from last week.",
    "Experian refuses to remove a late payment history that wasn't mine."
]

# Run prediction
predictions = pipeline.predict(incoming_complaints)
print(predictions)

['mortgages_and_loans' 'credit_reporting']


In [68]:
df.count()

,0
Unnamed: 0,124472
product,124472
narrative,124472
